<p align="center">
  <h1 align="center">🍳 Cookbook 01: Deep Learning Auto-Compression</h1>
  <p align="center">
    <strong>Strict Validation & Multi-Model Benchmarking with GradTracer</strong>
  </p>
</p>

---

In this recipe, we demonstrate how to compress standard Hugging Face Transformer models using **GradTracer's Advanced Analyzers**.

### What we will cover:
1. **Multi-Model Benchmarking**: Testing GradTracer across 5 different HuggingFace models.
2. **CompressionTracker**: Goal-based auto-search for optimal pruning sparsity.
3. **QuantizationAdvisor**: Mixed-precision bit recommendations based on training dynamics.
4. **Strict Validation**: Explicitly comparing original vs. compressed model accuracy and size.

## 1. Setup Environment & Configuration

In [ ]:
# !pip install gradtracer torch transformers datasets scikit-learn numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from datasets import load_dataset
from torch.utils.data import DataLoader
import copy, json, time

from gradtracer import FlowTracker
from gradtracer.analyzers.compression import CompressionTracker
from gradtracer.analyzers.quantization import QuantizationAdvisor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")

# --- CONFIGURATION ---
CONFIG = {
    "dataset_name": "rotten_tomatoes",     
    "max_length": 128,
    "batch_size": 32,
    "performance_floor": 0.95, # Keep at least 95% of original accuracy
    "profile_steps": 40
}

# 5 Popular HuggingFace Models for Benchmarking
MODELS_TO_TEST = [
    "distilbert-base-uncased", 
    "bert-base-uncased",
    "albert-base-v2",
    "roberta-base",
    "google/electra-small-discriminator"
]

## 2. Utility Functions (Evaluation & Tracking)

In [ ]:
def evaluate_accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            inputs = {'input_ids': batch['input_ids'].to(device), 'attention_mask': batch['attention_mask'].to(device)}
            outputs = model(**inputs)
            correct += (outputs.logits.argmax(dim=-1) == batch['label'].to(device)).sum().item()
            total += batch['label'].size(0)
    return correct / total

def profile_model(model, train_loader):
    """Run a quick training loop to gather dynamics with FlowTracker"""
    tracker = FlowTracker(model, track_gradients=True, track_interval=5)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
    model.train()
    
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        loss = model(batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device), labels=batch['label'].to(device)).loss
        loss.backward()
        tracker.step(loss.item())
        optimizer.step()
        if i >= CONFIG['profile_steps']: break
        
    return tracker


## 3. Multi-Model Compression Benchmark & Validation
We loop through 5 different architectures, profiling them dynamically, and using `CompressionTracker` to find the optimal sparsity that maintains 95% performance.

In [ ]:
results_summary = []

for model_name in MODELS_TO_TEST:
    print("=" * 70)
    print(f"🚀 Benchmarking Model: {model_name}")
    print("=" * 70)
    
    # 1. Load Model & Data
    try:
        print(f"  -> Loading {model_name}...")
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
        tokenizer = AutoTokenizer.from_pretrained(model_name)
    except Exception as e:
        print(f"  [!] Failed to load {model_name}: {e}")
        continue
        
    ds = load_dataset(CONFIG["dataset_name"])
    # Quick hack to make all tokenizers pad/truncate nicely
    tokenized_ds = ds.map(lambda x: tokenizer(x["text"], padding="max_length", truncation=True, max_length=CONFIG["max_length"]), batched=True)
    tokenized_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    train_loader = DataLoader(tokenized_ds["train"], batch_size=CONFIG["batch_size"], shuffle=True)
    test_loader = DataLoader(tokenized_ds["validation" if "validation" in tokenized_ds else "test"], batch_size=CONFIG["batch_size"]*2)
    
    # 2. Establish Baseline Accuracy
    base_acc = evaluate_accuracy(model, test_loader)
    print(f"  ✅ Baseline Accuracy: {base_acc*100:.2f}%")
    
    # 3. Profile Dynamics
    print("  -> Profiling dynamics with FlowTracker...")
    tracker = profile_model(model, train_loader)
    
    # 4. Advanced Compression Search using CompressionTracker
    print(f"  -> Running Auto-Compression Search (Target: >= {CONFIG['performance_floor']*100}% of baseline)...")
    # Use a lambda to evaluate the model during search
    eval_fn = lambda m: evaluate_accuracy(m, test_loader)
    comp_tracker = CompressionTracker(model, eval_fn=eval_fn)
    
    # GradTracer finds the optimal config automatically
    # We use a binary search strategy for speed
    res = comp_tracker.auto_compress(
        method="pruning", 
        performance_floor=CONFIG["performance_floor"], 
        search_range=(0.1, 0.7),
        search_strategy="binary",
        precision=0.1
    )
    
    print(f"\n  🏆 Optimal Compression Found!")
    print(f"     - Sparsity: {res.optimal_config.get('sparsity', 0)*100:.1f}%")
    print(f"     - Performance: {res.performance_compressed*100:.2f}%")
    print(f"     - Size: {res.size_original_mb:.1f}MB -> {res.size_compressed_mb:.1f}MB ({res.size_reduction*100:.1f}% reduction)")
    
    # 5. Strict Validation on Optimal Config
    print("\n  -> Strict Validation on Compressed Model...")
    optimal_sparsity = res.optimal_config.get('sparsity', 0)
    
    if optimal_sparsity > 0:
        comp_tracker._apply_pruning(model, optimal_sparsity)
        
    final_acc = evaluate_accuracy(model, test_loader)
    retained = (final_acc / base_acc) * 100 if base_acc > 0 else 0
    
    print(f"  ✅ Validated Accuracy: {final_acc*100:.2f}% ({retained:.1f}% retained)")
    if retained >= (CONFIG['performance_floor'] * 100 - 1.0):
        print("  🟢 VALIDATION PASSED ✓")
    else:
        print("  🔴 VALIDATION FAILED ✗")
        
    # 6. Quantization Advisor (Mixed Precision)
    print("\n  -> Quantization Advisor (Mixed-Precision)...")
    q_advisor = QuantizationAdvisor(tracker)
    q_red = q_advisor.estimated_size_reduction()
    print(f"     - Recommended Avg Precision: {q_red['avg_bits']:.1f}-bit")
    print(f"     - Est. Quantization Size Reduction: {q_red['estimated_reduction_pct']:.1f}%")
    
    results_summary.append({
        "model": model_name,
        "base_acc": base_acc,
        "final_acc": final_acc,
        "retained": retained,
        "sparsity": optimal_sparsity,
        "size_reduction": res.size_reduction,
        "avg_bits": q_red['avg_bits']
    })
    print("\n")


## 4. Final Summary Report

In [ ]:
print("=" * 105)
print(f" {'Model':<35} | {'Base Acc':<9} | {'Comp Acc':<9} | {'Retained':<9} | {'Optimal Sp':<10} | {'Sze Reduce':<10} | {'Avg Bits'}")
print("-" * 105)
for r in results_summary:
    print(f" {r['model']:<35} | {r['base_acc']*100:>8.2f}% | {r['final_acc']*100:>8.2f}% | {r['retained']:>8.1f}% | {r['sparsity']*100:>9.1f}% | {r['size_reduction']*100:>9.1f}% | {r['avg_bits']:>5.1f}-b")
print("=" * 105)